# AN-RA io.net Training Notebook

Run on io.net H100/A100/RTX6000Ada GPU nodes.
No Google Drive. Checkpoints go to S3/Wasabi.

**Prerequisites:** Set S3 credentials in CELL 1 before running.


In [ ]:
# CELL 1 — CONFIG
# Edit these before running. Everything else runs top-to-bottom.

# ── Repo ──────────────────────────────────────────────────────────────
REPO_URL  = 'https://github.com/dhurv0045com-spec/An-Ra-the-new-AGI.git'
REPO_PATH = '/workspace/An-Ra-the-new-AGI'

# ── S3 / Wasabi checkpoint storage ─────────────────────────────────────
# Set these or export as environment variables before running
S3_BUCKET       = 'anra-checkpoints'       # your bucket name
S3_PREFIX       = 'v2'                      # folder inside bucket
S3_ENDPOINT_URL = 'https://s3.wasabisys.com'  # or AWS, Backblaze, etc.
AWS_ACCESS_KEY  = ''   # ← fill in or use env var AWS_ACCESS_KEY_ID
AWS_SECRET_KEY  = ''   # ← fill in or use env var AWS_SECRET_ACCESS_KEY

# ── Training config ────────────────────────────────────────────────────
MODEL_SIZE       = '1b'       # '25m' (dev) or '1b' (frontier)
TRAINING_MODE    = 'session'  # 'status' | 'session' | 'train' | 'eval'
SESSION_MINUTES  = 240        # io.net sessions can run longer than Colab
OUROBOROS_MINUTES = 15
RUN_TESTS        = False

# ── io.net machine info ────────────────────────────────────────────────
# These are set automatically — just for display
import subprocess, sys, os
try:
    import torch
    GPU_NAME  = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
    GPU_MEM   = f'{torch.cuda.get_device_properties(0).total_memory / 1e9:.0f} GB' if torch.cuda.is_available() else 'N/A'
except Exception:
    GPU_NAME, GPU_MEM = 'unknown', 'unknown'

import psutil
RAM_GB = f'{psutil.virtual_memory().total / 1e9:.0f} GB'
CPU_CORES = os.cpu_count()

print('AN-RA IO.NET SESSION')
print('=' * 60)
print(f'GPU:             {GPU_NAME}')
print(f'GPU Memory:      {GPU_MEM}')
print(f'System RAM:      {RAM_GB}')
print(f'CPU cores:       {CPU_CORES}')
print(f'Model:           {MODEL_SIZE}')
print(f'Mode:            {TRAINING_MODE}')
print(f'Session minutes: {SESSION_MINUTES}')
print(f'S3 bucket:       {S3_BUCKET}/{S3_PREFIX}')

# Validate GPU is present for 1B training
if MODEL_SIZE == '1b' and (not torch.cuda.is_available() or 'CPU' in GPU_NAME):
    raise RuntimeError('1B model requires a CUDA GPU. Switch to 25m or request a GPU node.')
if MODEL_SIZE == '1b':
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    if vram_gb < 20:
        print(f'WARNING: {vram_gb:.0f}GB VRAM may be tight for 1B training. Gradient checkpointing will activate.')


In [ ]:
# CELL 2 — INSTALL DEPENDENCIES
import subprocess, sys

print('Installing dependencies...')
packages = [
    'torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121',
    'tokenizers transformers accelerate',
    'faiss-gpu',  # GPU FAISS for io.net
    'sentence-transformers',
    'sympy scipy networkx psutil gitpython tqdm',
    'fastapi uvicorn httpx',
    'boto3',  # S3 checkpoint sync
    'git+https://github.com/KellerJordan/Muon',  # Muon optimizer
]
for pkg in packages:
    result = subprocess.run(
        f'{sys.executable} -m pip install -q {pkg}',
        shell=True, capture_output=True, text=True
    )
    if result.returncode != 0:
        print(f'WARN: {pkg} failed: {result.stderr[-200:]}')
    else:
        print(f'  OK: {pkg.split()[0]}')
print('Done.')


In [ ]:
# CELL 3 — REPO SETUP
import os, shutil, subprocess

print('REPOSITORY')
print('=' * 60)
if os.path.isdir(REPO_PATH):
    result = subprocess.run(['git', 'pull', '--ff-only'], cwd=REPO_PATH, capture_output=True, text=True)
    if result.returncode != 0:
        print('Pull failed; recloning...')
        shutil.rmtree(REPO_PATH)
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, REPO_PATH], check=True)
    else:
        print('Repo updated:', result.stdout.strip())
else:
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, REPO_PATH], check=True)
    print('Repo cloned to', REPO_PATH)

import sys
if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)
os.chdir(REPO_PATH)
print('Working dir:', os.getcwd())
print('Git commit:', subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], cwd=REPO_PATH, capture_output=True, text=True).stdout.strip())


In [ ]:
# CELL 4 — RESTORE CHECKPOINTS FROM S3
import boto3, os, pathlib

print('CHECKPOINT RESTORE FROM S3')
print('=' * 60)

# Configure credentials
access_key = AWS_ACCESS_KEY or os.environ.get('AWS_ACCESS_KEY_ID', '')
secret_key = AWS_SECRET_KEY or os.environ.get('AWS_SECRET_ACCESS_KEY', '')

CHECKPOINT_FILES = ['anra_v2_brain.pt', 'anra_v2_identity.pt', 'anra_v2_ouroboros.pt']
restored = []

if not access_key:
    print('No S3 credentials — starting from scratch (fresh model init).')
    print('Set AWS_ACCESS_KEY_ID and AWS_SECRET_ACCESS_KEY env vars to restore.')
else:
    s3 = boto3.client(
        's3',
        endpoint_url=S3_ENDPOINT_URL,
        aws_access_key_id=access_key,
        aws_secret_access_key=secret_key,
    )
    for ckpt in CHECKPOINT_FILES:
        s3_key = f'{S3_PREFIX}/{ckpt}'
        local_path = pathlib.Path(REPO_PATH) / ckpt
        try:
            s3.download_file(S3_BUCKET, s3_key, str(local_path))
            size_mb = local_path.stat().st_size / 1e6
            print(f'  Restored: {ckpt} ({size_mb:.0f} MB)')
            restored.append(ckpt)
        except Exception as exc:
            print(f'  Not found: {ckpt} ({exc})')

    # Also restore training data if not present
    local_data = pathlib.Path(REPO_PATH) / 'training_data' / 'anra_training.txt'
    if not local_data.exists():
        try:
            s3.download_file(S3_BUCKET, f'{S3_PREFIX}/anra_training.txt', str(local_data))
            print(f'  Restored: training data ({local_data.stat().st_size / 1e6:.1f} MB)')
        except Exception:
            print('  Training data: using repo copy')

print(f'Checkpoints restored: {len(restored)}/{len(CHECKPOINT_FILES)}')


In [ ]:
# CELL 5 — DATA PREPARATION
# Merge any additional data sources. io.net has no Drive,
# so additional data should be in S3 or passed via the machine's local disk.
import os, glob, shutil

print('DATA PREPARATION')
print('=' * 60)

data_dir = os.path.join(REPO_PATH, 'training_data')
main_data = os.path.join(data_dir, 'anra_training.txt')

# Look for extra data files the user uploaded to the machine
extra_sources = [
    '/workspace/extra_data',        # user uploads to /workspace/extra_data/
    os.path.expanduser('~/anra_data'),
]
extra_files = []
for src_dir in extra_sources:
    if os.path.isdir(src_dir):
        extra_files.extend(glob.glob(f'{src_dir}/**/*.txt', recursive=True))
        extra_files.extend(glob.glob(f'{src_dir}/**/*.jsonl', recursive=True))

if extra_files:
    print(f'Found {len(extra_files)} extra data file(s) — merging into training data')
    with open(main_data, 'a', encoding='utf-8') as out:
        for ef in extra_files:
            out.write(f'\n')
            with open(ef, 'r', encoding='utf-8', errors='replace') as f:
                shutil.copyfileobj(f, out)
            size_kb = os.path.getsize(ef) / 1024
            print(f'  Merged: {os.path.basename(ef)} ({size_kb:.0f} KB)')
else:
    print('No extra data — using repo training_data/ only')
    print('TIP: Upload .txt/.jsonl files to /workspace/extra_data/ and rerun this cell')

total_mb = os.path.getsize(main_data) / 1e6
print(f'Training data total: {total_mb:.1f} MB')
if total_mb < 50:
    print('WARNING: < 50MB of training data. Model will overfit. Add more data for meaningful training.')


In [ ]:
# CELL 6 — PREFLIGHT + SYSTEM REPORT
import subprocess, sys, os

print('PREFLIGHT')
print('=' * 60)

env = os.environ.copy()
env['PYTHONPATH'] = REPO_PATH

# Apply component flags to feature_flags.json
flag_code = '\nfrom engine.feature_flags import set_flag\nflags = {\n    "brain": True, "tokenizer": True, "data_mix": True,\n    "training_loop": True, "evaluation": True, "runtime": True,\n    "identity": True, "memory": True, "phase2_memory": True,\n    "goals": True, "agent_loop": True, "master_system": True,\n    "self_improvement": True, "self_modification": True,\n    "ouroboros": True, "ghost_memory": True,\n    "symbolic_bridge": True, "sovereignty": True,\n}\nfor k, v in flags.items():\n    set_flag(k, v)\nprint("Feature flags applied")\n'
r = subprocess.run([sys.executable, '-c', flag_code], cwd=REPO_PATH, env=env, capture_output=True, text=True)
print(r.stdout)
if r.returncode != 0:
    print('Flag warning:', r.stderr[-500:])

# System report
r = subprocess.run([sys.executable, 'anra.py', '--report'], cwd=REPO_PATH, env=env, capture_output=True, text=True, timeout=60)
print(r.stdout[-3000:])
if r.returncode != 0:
    print('Report issues:', r.stderr[-1000:])

# Readiness check
r = subprocess.run([sys.executable, 'scripts/readiness.py'], cwd=REPO_PATH, env=env, capture_output=True, text=True, timeout=30)
print(r.stdout[-1000:])


In [ ]:
# CELL 7 — TRAINING RUN
import subprocess, sys, os, time

print('TRAINING RUN')
print('=' * 60)

env = os.environ.copy()
env['PYTHONPATH'] = REPO_PATH

cmd = [
    sys.executable, '-m', 'training.train_unified',
    '--mode', TRAINING_MODE,
    '--model-size', MODEL_SIZE,
]
if TRAINING_MODE not in {'status', 'eval'}:
    cmd += ['--session-minutes', str(SESSION_MINUTES),
            '--ouroboros_minutes', str(OUROBOROS_MINUTES)]

print('Command:', ' '.join(cmd))
start = time.time()

proc = subprocess.Popen(
    cmd, cwd=REPO_PATH, env=env,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1
)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()

TRAINING_EXIT_CODE = proc.returncode
elapsed = (time.time() - start) / 60
print(f'\nFinished: code={TRAINING_EXIT_CODE} elapsed={elapsed:.1f} min')
if TRAINING_EXIT_CODE not in (0, None):
    print('Training exited with non-zero code. Check output above.')


In [ ]:
# CELL 8 — COMPONENT SCORECARD
import subprocess, sys, os

print('COMPONENT SCORECARD')
print('=' * 60)

env = os.environ.copy()
env['PYTHONPATH'] = REPO_PATH

score_code = '\nfrom agents.supervisor import SupervisorAgent\nfrom engine.metric_bus import get_metric_bus\n\nbus = get_metric_bus()\nsnap = bus.snapshot()\ndeltas = getattr(bus, "_last_deltas", {})\nprint(f"MetricBus: {len(snap)} components tracked")\nfor comp, metrics in snap.items():\n    delta = deltas.get(comp, {})\n    score = metrics.get("score", "?")\n    d_str = f"  Δ={delta.get(\'score\',0):+.3f}" if delta.get("score") is not None else ""\n    print(f"  {comp:<28} score={score}{d_str}")\n'
r = subprocess.run([sys.executable, '-c', score_code], cwd=REPO_PATH, env=env, capture_output=True, text=True, timeout=30)
print(r.stdout)
if r.stderr:
    print(r.stderr[-500:])


In [ ]:
# CELL 9 — UPLOAD CHECKPOINTS TO S3
import boto3, os, pathlib, glob

print('CHECKPOINT SYNC TO S3')
print('=' * 60)

access_key = AWS_ACCESS_KEY or os.environ.get('AWS_ACCESS_KEY_ID', '')
secret_key = AWS_SECRET_KEY or os.environ.get('AWS_SECRET_ACCESS_KEY', '')

if not access_key:
    print('No S3 credentials — skipping upload. Set AWS_ACCESS_KEY_ID to enable.')
else:
    s3 = boto3.client('s3', endpoint_url=S3_ENDPOINT_URL,
                      aws_access_key_id=access_key,
                      aws_secret_access_key=secret_key)

    # Upload checkpoints
    for pattern in ['anra_v2_*.pt', '*.pt']:
        for fpath in glob.glob(os.path.join(REPO_PATH, pattern)):
            fname = os.path.basename(fpath)
            s3_key = f'{S3_PREFIX}/{fname}'
            size_mb = os.path.getsize(fpath) / 1e6
            try:
                s3.upload_file(fpath, S3_BUCKET, s3_key)
                print(f'  Uploaded: {fname} ({size_mb:.0f} MB) -> s3://{S3_BUCKET}/{s3_key}')
            except Exception as exc:
                print(f'  Upload failed: {fname}: {exc}')

    # Upload reports
    for report_dir in ['output/v2/reports', 'output/v2/eval', 'output/v2/scorecards']:
        full_dir = os.path.join(REPO_PATH, report_dir)
        if not os.path.isdir(full_dir):
            continue
        for fpath in glob.glob(f'{full_dir}/*.json'):
            fname = os.path.basename(fpath)
            s3_key = f'{S3_PREFIX}/reports/{fname}'
            try:
                s3.upload_file(fpath, S3_BUCKET, s3_key)
                print(f'  Report: {fname}')
            except Exception as exc:
                print(f'  Report upload failed: {fname}: {exc}')

    # Upload HAL state
    hal = os.path.join(REPO_PATH, 'state', 'hal_state.json')
    if os.path.exists(hal):
        s3.upload_file(hal, S3_BUCKET, f'{S3_PREFIX}/hal_state.json')
        print('  HAL state synced')

    # Upload training data if it grew
    data_path = os.path.join(REPO_PATH, 'training_data', 'anra_training.txt')
    s3.upload_file(data_path, S3_BUCKET, f'{S3_PREFIX}/anra_training.txt')
    print(f'  Training data synced ({os.path.getsize(data_path)/1e6:.1f} MB)')

    print('Done. S3 bucket:', S3_BUCKET)


In [ ]:
# CELL 10 — OPTIONAL: LAUNCH API SERVER
# Uncomment to launch the FastAPI server on io.net.
# Access via the io.net port-forwarding tunnel or ngrok.
import subprocess, sys, os

# import subprocess
# env = os.environ.copy()
# env['PYTHONPATH'] = REPO_PATH
# subprocess.Popen(
#     [sys.executable, 'app.py', '--host', '0.0.0.0', '--port', '8000'],
#     cwd=REPO_PATH, env=env
# )
# print('API running at http://0.0.0.0:8000')
# print('Expose with: ssh -L 8000:localhost:8000 <your-ionet-ssh-host>')

print('API launch is commented out. Uncomment to run.')
